# HW5: Neural Networks

## Part (a): Gradient calculation and gradient descent

This notebook provides starter code for the calculus parts and the image classification task.

### Part (a) - Gradient descent for sigmoid neuron

Single neuron: $\hat{y} = \sigma(wx + b)$, loss $L = \frac{1}{2}(\hat{y} - y)^2$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1 - s)

# Data: (x, y) = (2, 1), initial (w, b) = (0.5, 0), alpha = 0.5, 20 iterations
x, y = 2.0, 1.0
w, b = 0.5, 0.0
alpha = 0.5
n_iter = 20

losses = []
for i in range(n_iter):
    z = w * x + b
    y_hat = sigmoid(z)
    L = 0.5 * (y_hat - y) ** 2
    losses.append(L)
    
    # Gradients: dL/dw = (y_hat - y) * sigma'(z) * x, dL/db = (y_hat - y) * sigma'(z)
    dL_dyhat = y_hat - y
    dL_dz = dL_dyhat * sigmoid_prime(z)
    dw = dL_dz * x
    db = dL_dz
    
    w -= alpha * dw
    b -= alpha * db
    
    if (i + 1) % 5 == 0 or i == 0:
        print(f"Iter {i+1}: w={w:.4f}, b={b:.4f}, y_hat={y_hat:.4f}, L={L:.6f}")

print(f"\nFinal (w, b) = ({w:.4f}, {b:.4f})")
print(f"Final y_hat = {sigmoid(w*x+b):.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses, 'o-')
plt.xlabel('Iteration')
plt.ylabel('Loss L')
plt.title('Loss vs iteration (sigmoid neuron)')
plt.show()

## Part (b): Image classification with EuroSAT (smaller subset)

Use a **subset** of EuroSAT (3-4 classes, limited images) for faster training. Load via `torchvision.datasets.EuroSAT`, which downloads automatically.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

In [ ]:
# Load EuroSAT - downloads automatically (~90MB)
# Use subset for faster training: 4 classes, ~200 images per class
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.EuroSAT(root='../data', download=True, transform=transform)

# EuroSAT has 10 classes - use first 4 for quicker runs
n_classes = 4
class_indices = {i: [] for i in range(n_classes)}
for idx in range(len(full_dataset)):
    _, label = full_dataset[idx]
    if label < n_classes and len(class_indices[label]) < 250:
        class_indices[label].append(idx)

subset_indices = []
for c in range(n_classes):
    subset_indices.extend(class_indices[c])

dataset = Subset(full_dataset, subset_indices)
class_names = full_dataset.classes[:n_classes]
print(f"Using {len(dataset)} images from {n_classes} classes: {class_names}")

In [ ]:
# 80/20 train-test split
train_idx, test_idx = train_test_split(range(len(dataset)), test_size=0.2, stratify=[dataset[i][1] for i in range(len(dataset))], random_state=42)
train_dataset = Subset(dataset, train_idx)
test_dataset = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Simple CNN for 64x64 RGB images
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN(num_classes=n_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training
n_epochs = 10
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# Evaluate on test set
model.eval()
all_preds = []
all_labels = []
all_images = []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        preds = torch.argmax(outputs, 1)
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())
        all_images.extend(images.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
accuracy = accuracy_score(all_labels, all_preds)
print(f"Test accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

In [ ]:
# Sample images with predictions
n_show = 12
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.flatten()
for i in range(min(n_show, len(all_images))):
    img = all_images[i].transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    img = np.clip(img, 0, 1)
    axes[i].imshow(img)
    axes[i].set_title(f'True: {class_names[all_labels[i]]}\nPred: {class_names[all_preds[i]]}')
    axes[i].axis('off')
plt.suptitle('Sample Test Images: Actual vs Predicted')
plt.tight_layout()
plt.show()